<a href="https://colab.research.google.com/github/anjaliii210/LLM-Post-Training-with-QLoRA-SFT-DPO/blob/main/3_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = "/content/drive/MyDrive/LLM_PostTraining"

print("Base folder exists:", os.path.exists(base))

if os.path.exists(base):
    print(os.listdir(base))

Mounted at /content/drive
Base folder exists: True
['base_model', 'qlora_adapter', 'preference_data', 'sft_adapter', 'dpo_adapter']


In [ ]:
!find /content/drive/MyDrive -name "adapter_config.json"

/content/drive/MyDrive/LLM_PostTraining/qlora_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/sft_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/ref/adapter_config.json


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.9 MB/s  0:00:00


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

PyTorch: 2.14.1+cu130
CUDA: 13.0


In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.6 MB/s  0:00:00


In [ ]:
import os
import torch
import inspect
import bitsandbytes as bnb

from datasets import load_from_disk, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    PeftModel,
    prepare_model_for_kbit_training
)
from trl import DPOTrainer, DPOConfig

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("TRL:", __import__("trl").__version__)
print("PEFT:", __import__("peft").__version__)

print("\nDPOTrainer signature:")
print(inspect.signature(DPOTrainer.__init__))

print("\nDPOConfig signature:")
print(inspect.signature(DPOConfig))

PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4
TRL: 1.18.0
PEFT: 0.21.2

DPOTrainer signature:
(self, model: 'str | PreTrainedModel | PeftModel', ref_model: transformers.modeling_utils.PreTrainedModel | None = None, args: trl.trainer.dpo_config.DPOConfig | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callba

In [ ]:
from datasets import DatasetDict, load_dataset

raw_dataset = load_dataset("trl-lib/ultrafeedback_binarized")

# Reproduce the 800/100/100 split
train_pool = raw_dataset["train"].shuffle(seed=42)

train_raw = train_pool.select(range(800))
eval_raw = train_pool.select(range(800, 900))

# Use the dataset's official test split
test_raw = raw_dataset["test"].select(range(100))

print(len(train_raw), len(eval_raw), len(test_raw))

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

800 100 100


from google.colab import drive
drive.mount("/content/drive")

SAVE_PATH = "/content/drive/MyDrive/LLM_PostTraining/preference_data"

preference_data = DatasetDict({
    "train": train_raw,
    "validation": eval_raw,
    "test": test_raw
})

preference_data.save_to_disk(SAVE_PATH)

print("Preference dataset saved!")

In [ ]:
raw_dataset = load_dataset("trl-lib/ultrafeedback_binarized")

dataset = raw_dataset["train"].shuffle(seed=42).select(range(1000))

train_raw = dataset.select(range(800))
eval_raw = dataset.select(range(800, 900))
test_raw = dataset.select(range(900, 1000))

print(len(train_raw), len(eval_raw), len(test_raw))

800 100 100


In [ ]:
from datasets import load_from_disk

preference_data = load_from_disk(SAVE_PATH)

print(preference_data)
print(preference_data["train"][0])

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
        num_rows: 100
    })
    test: Dataset({
        features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
        num_rows: 100
    })
})
{'chosen': [{'content': 'INPUT ARTICLE: Article: Different laptop models have different numeric pad configurations.  If your U, I, and O keys have 4, 5, and 6 printed in the lower corner, you have an older laptop with an alternate numeric pad. See the next section for details on using it. The ThinkPad line of laptops do not use an alternate numeric pad. You\'ll need to use the method in this section as a workaround. Some larger models have a dedicated numeric pad. Click the "Start" button in the lower-right corner of the desktop. In many versions of Windows, this is just a Windows icon. The Sta

In [ ]:
def convert_to_dpo(example):
    chosen = example["chosen"]
    rejected = example["rejected"]

    # The prompt is everything before the final assistant response.
    prompt = chosen[:-1]

    # The final assistant messages are the two competing responses.
    chosen_response = chosen[-1:]
    rejected_response = rejected[-1:]

    # Both responses must belong to the same prompt.
    if chosen[:-1] != rejected[:-1]:
        raise ValueError("Chosen and rejected prompts do not match.")

    if chosen[-1]["role"] != "assistant":
        raise ValueError("Chosen completion is not an assistant message.")

    if rejected[-1]["role"] != "assistant":
        raise ValueError("Rejected completion is not an assistant message.")

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response
    }


dpo_train = preference_data["train"].map(
    convert_to_dpo,
    remove_columns=preference_data["train"].column_names
)

dpo_eval = preference_data["validation"].map(
    convert_to_dpo,
    remove_columns=preference_data["validation"].column_names
)

dpo_test = preference_data["test"].map(
    convert_to_dpo,
    remove_columns=preference_data["test"].column_names
)

print(dpo_train)
print(dpo_train[0])

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 800
})
{'chosen': [{'content': 'Reach out to friends, family, or significant others when feeling lonely at night. Engage in social activities such as dinners, movies, or other events with friends. Focus your energy on evening activities or join classes to add structure to your routine. Find groups on meetup.com to connect with people who share your interests. Volunteer at local organizations to shift your focus from your own loneliness and help others in need. Consider visiting elderly or sick relatives to uplift your perspective.\n\nConfidence: 95%', 'role': 'assistant'}], 'rejected': [{'content': "The article provides suggestions for ways to combat loneliness at night. It encourages readers to reach out to friends and family members, or to engage in activities that they enjoy, such as writing, painting, or volunteering. The article also suggests that readers invite people over to their homes, or go out to events 

In [ ]:
sample = dpo_train[0]

print("PROMPT")
for msg in sample["prompt"]:
    print(msg["role"], ":", msg["content"])

print("\nCHOSEN")
for msg in sample["chosen"]:
    print(msg["role"], ":", msg["content"])

print("\nREJECTED")
for msg in sample["rejected"]:
    print(msg["role"], ":", msg["content"])

PROMPT
user : INPUT ARTICLE: Article: Different laptop models have different numeric pad configurations.  If your U, I, and O keys have 4, 5, and 6 printed in the lower corner, you have an older laptop with an alternate numeric pad. See the next section for details on using it. The ThinkPad line of laptops do not use an alternate numeric pad. You'll need to use the method in this section as a workaround. Some larger models have a dedicated numeric pad. Click the "Start" button in the lower-right corner of the desktop. In many versions of Windows, this is just a Windows icon. The Start menu will appear above the button. If you are using Windows 8 and don't see the Start button, press ⊞ Win on the keyboard. This will open the Start screen. You can start typing immediately when the Start menu or screen is open to begin searching. You'll see "On-Screen Keyboard" in the search results. If your Lenovo laptop does not have a NumLock key, you can use the Windows On-Screen Keyboard to access th

Loading SFT Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = "/content/drive/MyDrive/LLM_PostTraining"

print("Base folder exists:", os.path.exists(base))

if os.path.exists(base):
    print(os.listdir(base))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base folder exists: True
['base_model', 'qlora_adapter', 'preference_data', 'sft_adapter', 'dpo_adapter']


In [ ]:
!find /content/drive/MyDrive -name "adapter_config.json"

/content/drive/MyDrive/LLM_PostTraining/qlora_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/sft_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/ref/adapter_config.json


In [ ]:
!find /content/drive/MyDrive/LLM_PostTraining \
  -name "adapter_config.json" -o -name "adapter_model.safetensors"

/content/drive/MyDrive/LLM_PostTraining/qlora_adapter/adapter_model.safetensors
/content/drive/MyDrive/LLM_PostTraining/qlora_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/sft_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/sft_adapter/adapter_model.safetensors
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/adapter_model.safetensors
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/ref/adapter_model.safetensors
/content/drive/MyDrive/LLM_PostTraining/dpo_adapter/ref/adapter_config.json


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

SFT_ADAPTER_PATH = (
    "/content/drive/MyDrive/LLM_PostTraining/sft_adapter"
)

DPO_SAVE_PATH = (
    "/content/drive/MyDrive/LLM_PostTraining/dpo_adapter"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True
)

model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER_PATH,
    is_trainable=True
)

print("SFT adapter loaded.")
print("Trainable parameters:")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

SFT adapter loaded.
Trainable parameters:
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
trainable = [
    name for name, param in model.named_parameters()
    if param.requires_grad
]

print("Number of trainable tensors:", len(trainable))
print("\nSample trainable parameters:")
for name in trainable[:10]:
    print(name)

Number of trainable tensors: 192

Sample trainable parameters:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight


In [ ]:
from collections import Counter

trainable_dtypes = Counter(
    str(param.dtype)
    for param in model.parameters()
    if param.requires_grad
)

print(trainable_dtypes)

Counter({'torch.float32': 192})


In [ ]:
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

DPO Configuration

In [ ]:
dpo_args = DPOConfig(
    output_dir="./dpo_checkpoints",

    beta=0.1,
    loss_type="sigmoid",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=512,

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    report_to="none"
)

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer
)

In [ ]:
print("PEFT adapters:")
print(trainer.model.peft_config.keys())

print("\nTrainable parameters after trainer initialization:")
trainer.model.print_trainable_parameters()

from collections import Counter

print("\nTrainable dtypes:")
print(Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

PEFT adapters:
dict_keys(['default', 'ref'])

Trainable parameters after trainer initialization:
trainable params: 2,162,688 || all params: 498,358,144 || trainable%: 0.4340

Trainable dtypes:
Counter({'torch.bfloat16': 192})


In [ ]:
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
        if param.grad is not None:
            param.grad.data = param.grad.data.float()

training_result = trainer.train()
print(training_result)

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.684966,0.686669,1.344193,494567.000000,-1.837953,-1.763678,0.668800,-0.022421,-0.036597,0.563830,0.014175,-302.751651,-250.059005


TrainOutput(global_step=93, training_loss=0.6835158973611811, metrics={'train_runtime': 465.0882, 'train_samples_per_second': 1.587, 'train_steps_per_second': 0.2, 'total_flos': 1215071446026240.0, 'train_loss': 0.6835158973611811, 'epoch': 1.0})


In [ ]:
trainer.model.save_pretrained(DPO_SAVE_PATH)
tokenizer.save_pretrained(DPO_SAVE_PATH)

print("DPO adapter saved:", DPO_SAVE_PATH)

DPO adapter saved: /content/drive/MyDrive/LLM_PostTraining/dpo_adapter


In [ ]:
eval_results = trainer.evaluate()

print(eval_results)

Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.684966,0.686669,1,1.344193,494567.000000,-1.837953,-1.763678,0.668800,-0.022421,-0.036597,0.563830,0.014175,-302.751651,-250.059005


{'eval_loss': 0.6866686344146729, 'eval_entropy': 1.3441928068691111, 'eval_num_tokens': 494567.0, 'eval_logits/chosen': -1.8379526428221544, 'eval_logits/rejected': -1.7636784062941615, 'eval_mean_token_accuracy': 0.6688003099345147, 'eval_rewards/chosen': -0.022421489394780128, 'eval_rewards/rejected': -0.036596776720661514, 'eval_rewards/accuracies': 0.5638297872340425, 'eval_rewards/margins': 0.014175287378515969, 'eval_logps/chosen': -302.75165072907794, 'eval_logps/rejected': -250.0590049079124}


In [ ]:
print("Global steps:", trainer.state.global_step)
print("Max steps:", trainer.state.max_steps)

for log in trainer.state.log_history:
    print(log)

Global steps: 93
Max steps: 93
{'loss': 0.6935484886169434, 'grad_norm': 12.320002555847168, 'learning_rate': 4.000000000000001e-06, 'entropy': 1.3183961741626262, 'num_tokens': 55947.0, 'logits/chosen': -1.6052230580232763, 'logits/rejected': -1.601330251299316, 'mean_token_accuracy': 0.6771926406770945, 'rewards/chosen': -0.0011264066401849959, 'rewards/rejected': -0.0003530171623424394, 'rewards/accuracies': 0.4875, 'rewards/margins': -0.0007733895039564231, 'logps/chosen': -273.2259510755539, 'logps/rejected': -267.98558351993563, 'epoch': 0.10840108401084012, 'step': 10}
{'loss': 0.6879833698272705, 'grad_norm': 11.597413063049316, 'learning_rate': 4.912761632854834e-06, 'entropy': 1.3054288238286973, 'num_tokens': 107619.0, 'logits/chosen': -1.7557533918832164, 'logits/rejected': -1.7062359661643651, 'mean_token_accuracy': 0.6804885409772397, 'rewards/chosen': 0.0067214937751487016, 'rewards/rejected': -0.0037089922127051976, 'rewards/accuracies': 0.725, 'rewards/margins': 0.0104

In [ ]:
import os

SFT_PATH = "/content/drive/MyDrive/LLM_PostTraining/sft_adapter"
DPO_PATH = "/content/drive/MyDrive/LLM_PostTraining/dpo_adapter"

for name, path in [("SFT", SFT_PATH), ("DPO", DPO_PATH)]:
    print(f"\n{name} adapter")
    for file in ["adapter_config.json", "adapter_model.safetensors"]:
        full_path = os.path.join(path, file)
        if os.path.exists(full_path):
            print(f"{file}: {os.path.getsize(full_path) / (1024**2):.2f} MB")
        else:
            print(f"MISSING: {file}")


SFT adapter
adapter_config.json: 0.00 MB
adapter_model.safetensors: 8.27 MB

DPO adapter
adapter_config.json: 0.00 MB
adapter_model.safetensors: 8.27 MB


In [ ]:
import json

# Inspect training history
for log in trainer.state.log_history:
    print(log)

# Save training logs to Google Drive
LOG_PATH = "/content/drive/MyDrive/LLM_PostTraining/dpo_training_logs.json"

with open(LOG_PATH, "w") as f:
    json.dump(trainer.state.log_history, f, indent=4)

print("Logs saved:", LOG_PATH)

{'loss': 0.6935484886169434, 'grad_norm': 12.320002555847168, 'learning_rate': 4.000000000000001e-06, 'entropy': 1.3183961741626262, 'num_tokens': 55947.0, 'logits/chosen': -1.6052230580232763, 'logits/rejected': -1.601330251299316, 'mean_token_accuracy': 0.6771926406770945, 'rewards/chosen': -0.0011264066401849959, 'rewards/rejected': -0.0003530171623424394, 'rewards/accuracies': 0.4875, 'rewards/margins': -0.0007733895039564231, 'logps/chosen': -273.2259510755539, 'logps/rejected': -267.98558351993563, 'epoch': 0.10840108401084012, 'step': 10}
{'loss': 0.6879833698272705, 'grad_norm': 11.597413063049316, 'learning_rate': 4.912761632854834e-06, 'entropy': 1.3054288238286973, 'num_tokens': 107619.0, 'logits/chosen': -1.7557533918832164, 'logits/rejected': -1.7062359661643651, 'mean_token_accuracy': 0.6804885409772397, 'rewards/chosen': 0.0067214937751487016, 'rewards/rejected': -0.0037089922127051976, 'rewards/accuracies': 0.725, 'rewards/margins': 0.01043048599094618, 'logps/chosen': 